### Environment Helpers

Small post-ingestion environment-population helpers that need to run after
`Canonical_Data` (catalog + schemas exist) and `Document_Data` (PDFs are in
volumes) but are too small or too miscellaneous to warrant their own stage.

**Current helpers**:

- **AI SQL demo on food-safety inspection PDFs** — exercises
  `ai_parse_document`, `ai_classify`, `ai_extract`, and `ai_summarize`
  against a small sample (`PDF_SAMPLE_SIZE` rows) from
  `/Volumes/{CATALOG}/food_safety/reports/` and writes the results to
  `{CATALOG}.food_safety.ai_*_inspections` tables for inspection from the
  SQL editor / a downstream dashboard.
- **ABAC governance** — registers four governed tag policies, applies
  the tags to columns/tables, creates four UDFs, and runs four
  `CREATE POLICY` statements that drive PII masking, region-based row
  filtering, high-value-order gating, and a regulated-document deny on
  `food_safety.ai_extracted_inspections`. See the markdown intro below
  the AI SQL section for the policy matrix.
- **Discover Domain tagging** — creates three governed tag policies
  (`caspers_domain_operations`, `caspers_domain_revenue`,
  `caspers_domain_compliance`) via `w.tag_policies` and applies them to
  every UC securable (catalogs / schemas / tables) the demo materialises,
  plus the five `all`-target AI/BI dashboards via
  `w.workspace_entity_tag_assignments`.  Genie spaces and Apps are tagged
  by their own stages (`Genie_Spaces`, `Databricks_App_Refund_Manager`,
  `Operational_App`).  All shared helpers live in `utils/domain_tags.py`.
  The three Discover Domains themselves are still UI-only — see
  `demos/dais2026-runbooks/SETUP.ipynb` §4.

Add new helpers in their own cells below.  Keep each idempotent
(`CREATE OR REPLACE`, `CREATE … IF NOT EXISTS`, `MERGE`) and best-effort
(skip gracefully if the inputs it needs are missing) so this stage can be
re-run safely as part of a `bundle run caspers`, and so it can be added to
targets that don't run `Document_Data` without breaking them.

In [ ]:
%pip install --upgrade databricks-sdk

In [ ]:
dbutils.library.restartPython()

In [ ]:
CATALOG = dbutils.widgets.get("CATALOG")

# Size of the AI SQL demo sample.  `ai_parse_document` bills per page and
# `ai_classify`/`ai_extract`/`ai_summarize` bill per row, so keep this small
# unless you are deliberately running a wider sweep.
PDF_SAMPLE_SIZE = 6
PDF_VOLUME_PATH = f"/Volumes/{CATALOG}/food_safety/reports"

print(f"environment_helpers: catalog={CATALOG}")
print(f"environment_helpers: PDF sample size for AI SQL demo = {PDF_SAMPLE_SIZE}")
print(f"environment_helpers: PDF volume path = {PDF_VOLUME_PATH}")

#### AI SQL demo on inspection PDFs

Reads `PDF_SAMPLE_SIZE` PDFs out of the food-safety `reports` volume and
chains four AI SQL functions over them:

| Function | What we get out | Output table |
|---|---|---|
| `ai_parse_document` | Per-PDF structured `VARIANT` + flattened text | `food_safety.ai_parsed_inspections` |
| `ai_classify` | Bucketed overall risk + dominant violation category | `food_safety.ai_classified_inspections` |
| `ai_extract` | Structured fields (date, location, score, grade) | `food_safety.ai_extracted_inspections` |
| `ai_summarize` | Short plain-English summary per inspection | `food_safety.ai_summarized_inspections` |

All cells are idempotent (`CREATE OR REPLACE`) and gated by the volume
check below so this stage stays safe in targets that don't run
`Document_Data`.

In [ ]:
# Guard: skip the entire AI SQL demo if the food-safety PDF volume is
# absent or empty.  This lets Environment_Helpers be added to targets that
# don't run Document_Data without making the task fail.
try:
    pdf_entries = [
        f for f in dbutils.fs.ls(PDF_VOLUME_PATH)
        if f.name.lower().endswith(".pdf")
    ]
    pdf_count = len(pdf_entries)
except Exception as e:
    print(
        f"⏭️  Cannot access {PDF_VOLUME_PATH} "
        f"({type(e).__name__}: {e}) — skipping AI SQL demo."
    )
    dbutils.notebook.exit("ai_sql_demo: skipped (volume missing)")

if pdf_count == 0:
    print(f"⏭️  No PDFs in {PDF_VOLUME_PATH} — skipping AI SQL demo.")
    dbutils.notebook.exit("ai_sql_demo: skipped (volume empty)")

print(f"✅ Found {pdf_count} PDFs in {PDF_VOLUME_PATH}")
print(f"   Will process the first {min(pdf_count, PDF_SAMPLE_SIZE)} via AI SQL.")

In [ ]:
# ai_parse_document → one row per PDF, with:
#   - the full structured VARIANT (kept for power users)
#   - a flat `text` column that concatenates every text/heading/caption
#     element in document order (the input the downstream AI SQL calls use)
#   - a couple of metadata columns for sanity checks
#
# Cost note: ai_parse_document bills per page, so we LIMIT the source to
# PDF_SAMPLE_SIZE rows.  Remove the LIMIT (and reconsider the volume scan)
# for a production parse pass.
spark.sql(f"""
CREATE OR REPLACE TABLE {CATALOG}.food_safety.ai_parsed_inspections
COMMENT 'Inspection PDFs parsed with ai_parse_document; one row per PDF.'
AS
WITH parsed AS (
  SELECT
    element_at(split(path, '/'), -1)                       AS pdf_name,
    ai_parse_document(content, map('version', '2.0'))      AS doc
  FROM READ_FILES('{PDF_VOLUME_PATH}/', format => 'binaryFile')
  WHERE lower(path) LIKE '%.pdf'
  ORDER BY path
  LIMIT {PDF_SAMPLE_SIZE}
)
SELECT
  pdf_name,
  doc                                                       AS parsed,
  array_join(
    transform(
      filter(
        try_variant_get(
          doc,
          '$.document.elements',
          'ARRAY<STRUCT<type STRING, content STRING>>'
        ),
        e -> e.type IN ('text', 'title', 'section_header', 'caption')
      ),
      e -> e.content
    ),
    '\\n'
  )                                                         AS text,
  size(
    try_variant_get(doc, '$.document.pages', 'ARRAY<STRUCT<id INT>>')
  )                                                         AS num_pages,
  CAST(doc:metadata:file_metadata:file_size AS BIGINT)      AS file_size_bytes
FROM parsed
""")

print(f"✅ Created {CATALOG}.food_safety.ai_parsed_inspections")
display(
    spark.sql(f"""
        SELECT pdf_name, num_pages, file_size_bytes, left(text, 200) AS text_preview
        FROM {CATALOG}.food_safety.ai_parsed_inspections
        ORDER BY pdf_name
    """)
)

In [ ]:
# ai_classify → assign each PDF to one of a fixed label set.  Two passes:
#   - overall_risk: bucketed severity of the inspection as a whole
#   - dominant_category: which violation category dominates the report
#
# Labels are chosen to match the categories the generator actually emits
# (see data/inspections/generate_inspection_reports.py) so the demo is
# interpretable against the ground-truth violations table.
spark.sql(f"""
CREATE OR REPLACE TABLE {CATALOG}.food_safety.ai_classified_inspections
COMMENT 'Per-PDF risk bucket + dominant violation category from ai_classify.'
AS
SELECT
  pdf_name,
  ai_classify(
    text,
    ARRAY(
      'clean_pass',
      'minor_violations_only',
      'major_violations_present',
      'critical_violations_present'
    )
  ) AS overall_risk,
  ai_classify(
    text,
    ARRAY(
      'Temperature Control',
      'Cross-Contamination',
      'Personal Hygiene',
      'Sanitation',
      'Personnel',
      'Facilities',
      'Equipment',
      'Food Labeling',
      'Maintenance',
      'Pest Control',
      'Administrative',
      'Chemical Safety'
    )
  ) AS dominant_violation_category
FROM {CATALOG}.food_safety.ai_parsed_inspections
WHERE text IS NOT NULL
""")

print(f"✅ Created {CATALOG}.food_safety.ai_classified_inspections")
display(
    spark.sql(f"""
        SELECT * FROM {CATALOG}.food_safety.ai_classified_inspections
        ORDER BY pdf_name
    """)
)

In [ ]:
# ai_extract → pull the structured fields the inspection PDFs print on
# their cover page out of the free-text the parser produced.  The result is
# a STRUCT<field STRING, ...> where each label in the array becomes a
# top-level field.
#
# These should track closely with the canonical values in
# {CATALOG}.food_safety.inspections — comparing the two is a quick way to
# eyeball how well the parse-then-extract chain is doing.
spark.sql(f"""
CREATE OR REPLACE TABLE {CATALOG}.food_safety.ai_extracted_inspections
COMMENT 'Structured fields lifted from each inspection PDF via ai_extract.'
AS
SELECT
  pdf_name,
  ai_extract(
    text,
    ARRAY(
      'inspection_date',
      'inspector_name',
      'location_name',
      'address',
      'score',
      'grade',
      'total_violations'
    )
  ) AS extracted
FROM {CATALOG}.food_safety.ai_parsed_inspections
WHERE text IS NOT NULL
""")

print(f"✅ Created {CATALOG}.food_safety.ai_extracted_inspections")
display(
    spark.sql(f"""
        SELECT
          pdf_name,
          extracted.inspection_date,
          extracted.inspector_name,
          extracted.location_name,
          extracted.score,
          extracted.grade,
          extracted.total_violations
        FROM {CATALOG}.food_safety.ai_extracted_inspections
        ORDER BY pdf_name
    """)
)

In [ ]:
# ai_summarize → a short plain-English summary per PDF.  The second arg
# is the max-words target (the function treats it as a soft bound).
#
# Useful as a "what's in this report" tooltip in dashboards or as the
# first column a reviewer sees when triaging a stack of inspections.
spark.sql(f"""
CREATE OR REPLACE TABLE {CATALOG}.food_safety.ai_summarized_inspections
COMMENT 'Short plain-English summary of each inspection PDF via ai_summarize.'
AS
SELECT
  pdf_name,
  ai_summarize(text, 60) AS summary
FROM {CATALOG}.food_safety.ai_parsed_inspections
WHERE text IS NOT NULL
""")

print(f"✅ Created {CATALOG}.food_safety.ai_summarized_inspections")
display(
    spark.sql(f"""
        SELECT * FROM {CATALOG}.food_safety.ai_summarized_inspections
        ORDER BY pdf_name
    """)
)

print()
print("✅ AI SQL helper complete — tables written to "
      f"{CATALOG}.food_safety.ai_*_inspections")

#### ABAC governance (DAIS 2026 beat)

Sets up an Attribute-Based Access Control story across the Casper's data
estate. Each cell is best-effort and idempotent — missing securables, older
runtimes (ABAC needs DBR 16.4+), or missing demo groups will print a warning
and continue, never fail the stage. The ABAC story is:

| # | Policy | Where | Effect |
|---|---|---|---|
| 1 | PII column mask | `simulator.locations.address` | Replaces street address with `***` unless caller is in `caspers_pii_readers` |
| 2 | Region row filter | `simulator.locations` | Drops rows whose location-code's region the caller can't see |
| 3 | High-value order gate | `_security.high_value_orders_snapshot.order_revenue` | Hides revenue > $500 unless caller is `caspers_finance` / `caspers_managers` |
| 4 | Regulated documents | `food_safety.ai_extracted_inspections` | Hides all rows unless caller is `caspers_compliance` |

**Why a snapshot table for policy #3 instead of `lakeflow.gold_order_header` directly?**
Databricks rejects ABAC policies on tables defined within a Lakeflow declarative
pipeline (`ABAC_POLICIES_NOT_SUPPORTED`). The high-value-order gate therefore
binds to a non-pipeline snapshot table this stage materializes (`CREATE OR
REPLACE TABLE … AS SELECT * FROM lakeflow.gold_order_header`). The snapshot only
refreshes when `Environment_Helpers` re-runs — that's fine for the demo, where
the point is to show the policy in action against a representative row set.
Policy #1 was moved off `lakeflow.silver_order_items.customer_addr` to
`simulator.locations.address` for the same reason; the canonical `address`
column is real street-address PII loaded from `locations.parquet`.

The demo groups (`caspers_pii_readers`, `caspers_us_users`,
`caspers_emea_users`, `caspers_geo_admins`, `caspers_finance`,
`caspers_managers`, `caspers_compliance`) don't need to exist for these
policies to be created — `is_account_group_member` returns false for
non-existent groups, which means the masking/filtering simply applies to
*everyone* (workspace admins still bypass). Create the groups in the
Account console if you want to demonstrate the unmasked perspective from
a separate identity. See `demos/dais2026-runbooks/SETUP.ipynb`.

In [ ]:
# ABAC step 1/4: register governed tag policies, then apply the tags to the
# columns / tables that step 3's policies will match against.
#
# IMPORTANT: `CREATE POLICY ... MATCH COLUMNS has_tag(<key>)` and
# `CREATE POLICY ... WHEN has_tag_value(<key>, <val>)` REQUIRE the tag key to
# be REGISTERED as a governed tag policy via `w.tag_policies.create_tag_policy`.
# Plain SQL `ALTER ... SET TAGS` alone is NOT enough — UC will compile-fail
# with `Unknown tag policy key '<key>'` (verified in the
# `Environment_Helpers` task run on 2026-06-05).  The earlier comment here
# was wrong on this point.
#
# So this cell does two things, in order:
#   (a) Register the 4 governed tag policies with their allowed value sets.
#       Idempotent: a re-run hits the ALREADY_EXISTS path silently.
#   (b) Apply each governed tag to its target column or table via SQL DDL.
#       `SET TAGS` happily applies governed tags as long as the (key, value)
#       pair matches the policy's allowed values.
#
# Tag-key naming: underscores, NOT dots.  UC table/column tags reject
# `, . : / - = % & ? > <` and leading/trailing spaces in tag KEYS.

GOV_SCHEMA = f"{CATALOG}._security"

# Non-pipeline snapshot of gold_order_header for the high-value-order ABAC
# policy.  Databricks rejects ABAC policies on tables defined within a
# Lakeflow declarative pipeline (`ABAC_POLICIES_NOT_SUPPORTED`), so the
# policy can't bind directly to lakeflow.gold_order_header — see the
# markdown intro above for the full reasoning.  The snapshot lives in
# _security alongside the ABAC UDFs/policy artifacts; refresh only happens
# when Environment_Helpers re-runs.
HIGH_VALUE_SNAPSHOT = f"{GOV_SCHEMA}.high_value_orders_snapshot"

# Non-pipeline snapshot of simulator.locations so the geo-region row
# filter can bind to a non-critical-path table.  The live
# simulator.locations table is read by the ops-dashboard app's /api/revenue
# (LEFT JOIN against gold_*), and the row filter dropped every row for
# the app SP because the demo geo groups don't exist in fresh accounts —
# the dashboard would render empty.  Demo narrative still works on the
# snapshot: "region row filter on _security.locations_snapshot".
LOCATIONS_SNAPSHOT  = f"{GOV_SCHEMA}.locations_snapshot"

# (full_object_name, kind, col, tag_key, tag_value)
#   kind ∈ {"table_column", "table"}
TAG_TARGETS = [
    # PII on the canonical street-address column in simulator.locations.
    # Moved here from lakeflow.silver_order_items.customer_addr (which is a
    # Lakeflow-pipeline table and so can't carry ABAC policies).  The
    # `address` column is loaded as-is from data/canonical/canonical_dataset/
    # locations.parquet and contains real street addresses.
    (f"{CATALOG}.simulator.locations",                  "table_column", "address",       "caspers_pii",         "address"),
    # Region marker on the location code (sfo/nyc/chi/lax = US; lon/muc/ams/via = EMEA).
    # Bound to the snapshot table — see LOCATIONS_SNAPSHOT comment above for
    # why this is no longer on simulator.locations directly.
    (LOCATIONS_SNAPSHOT,                                "table_column", "location_code", "caspers_geo_region",  "true"),
    # Per-order revenue on the non-pipeline snapshot table — the column
    # the high-value policy gates on.  Snapshot is materialized below.
    (HIGH_VALUE_SNAPSHOT,                               "table_column", "order_revenue", "caspers_value_class", "high_value"),
    # Table-level sensitivity tag — entire AI-extracted inspections table is regulated.
    (f"{CATALOG}.food_safety.ai_extracted_inspections", "table",         None,           "caspers_sensitivity", "restricted"),
]

# Governed tag policy spec: each key gets a single allowed value (the one
# we apply below).  If you add a new value to TAG_TARGETS, add it here too.
TAG_POLICY_VALUES = {
    "caspers_pii":         ["address"],
    "caspers_geo_region":  ["true"],
    "caspers_value_class": ["high_value"],
    "caspers_sensitivity": ["restricted"],
}

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {GOV_SCHEMA} COMMENT 'Casper\\'s ABAC UDFs, policy artifacts, and demo snapshot tables.'")
print(f"\u2705 Schema ready: {GOV_SCHEMA}")

# Materialize the high-value-order snapshot table before tagging it.  Source
# is lakeflow.gold_order_header which is materialized by the Lakeflow
# pipeline; Environment_Helpers depends on Spark_Declarative_Pipeline so the
# pipeline is at least started, but the gold tables may still be empty/missing
# on a cold first run.  Best-effort: print and continue if the source table
# isn't ready yet — a subsequent re-run of Environment_Helpers will create it.
print()
print(f"\u2014 Materializing {HIGH_VALUE_SNAPSHOT} \u2014")
src_table = f"{CATALOG}.lakeflow.gold_order_header"
try:
    if not spark.catalog.tableExists(src_table):
        print(f"  \u26a0\ufe0f  source table {src_table} doesn't exist yet "
              f"(pipeline hasn't produced gold tables) \u2014 skipping snapshot. "
              f"Re-run Environment_Helpers after the pipeline has produced data.")
    else:
        spark.sql(f"""
        CREATE OR REPLACE TABLE {HIGH_VALUE_SNAPSHOT}
        COMMENT 'Non-pipeline snapshot of lakeflow.gold_order_header so an ABAC policy can bind to it. Refreshed only when Environment_Helpers runs.'
        AS SELECT * FROM {src_table}
        """)
        row_count = spark.table(HIGH_VALUE_SNAPSHOT).count()
        print(f"  \u2705 {HIGH_VALUE_SNAPSHOT} ({row_count} rows)")
except Exception as e:
    print(f"  \u274c FAILED to create snapshot: {str(e).splitlines()[0]}")
    print(f"     The high-value-order tag in step (b) will fail; policy in step 3 will create on an empty schema.")

# Non-pipeline snapshot of simulator.locations.  See the LOCATIONS_SNAPSHOT
# comment near the top of this cell for why the geo-region tag and policy
# were moved off the live simulator.locations table.
print()
print(f"\u2014 Materializing {LOCATIONS_SNAPSHOT} \u2014")
loc_src = f"{CATALOG}.simulator.locations"
try:
    if not spark.catalog.tableExists(loc_src):
        print(f"  \u26a0\ufe0f  source table {loc_src} doesn't exist yet "
              f"(canonical_data hasn't run) \u2014 skipping snapshot. "
              f"Re-run Environment_Helpers after Canonical_Data has loaded.")
    else:
        spark.sql(f"""
        CREATE OR REPLACE TABLE {LOCATIONS_SNAPSHOT}
        COMMENT 'Non-pipeline snapshot of simulator.locations so the geo-region ABAC policy can bind to it without filtering rows out of the live ops-dashboard query path. Refreshed only when Environment_Helpers runs.'
        AS SELECT * FROM {loc_src}
        """)
        loc_rows = spark.table(LOCATIONS_SNAPSHOT).count()
        print(f"  \u2705 {LOCATIONS_SNAPSHOT} ({loc_rows} rows)")
except Exception as e:
    print(f"  \u274c FAILED to create snapshot: {str(e).splitlines()[0]}")
    print(f"     The geo-region tag in step (b) will fail; policy in step 3 will create on an empty schema.")

# Cleanup: drop the previous geo-region binding on the live simulator.locations
# (older deploys tagged the live table; we now tag the snapshot instead so the
# ops dashboard's /api/revenue isn't filtered out for the app SP).
# Best-effort: ignore failures on fresh deployments where there's nothing to drop.
print()
print("\u2014 Cleaning up legacy bindings on simulator schema \u2014")
for cleanup_sql in [
    f"ALTER TABLE {CATALOG}.simulator.locations ALTER COLUMN location_code UNSET TAGS ('caspers_geo_region')",
    f"DROP POLICY IF EXISTS caspers_region_filter ON SCHEMA {CATALOG}.simulator",
]:
    try:
        spark.sql(cleanup_sql)
        print(f"  \u2705 {cleanup_sql.split(chr(10))[0][:80]}")
    except Exception as e:
        msg = str(e).splitlines()[0]
        if any(s in msg for s in ("does not exist", "DOES_NOT_EXIST", "not found", "NOT_FOUND")):
            print(f"  \u2192 nothing to clean up: {cleanup_sql.split(chr(10))[0][:60]}")
        else:
            print(f"  \u26a0\ufe0f  cleanup failed (non-fatal): {msg}")

# (a) Register governed tag policies.
print()
print("\u2014 Registering governed tag policies \u2014")
try:
    from databricks.sdk import WorkspaceClient
    from databricks.sdk.service.tags import TagPolicy, Value
    _w_tags = WorkspaceClient()
    _tag_policy_api_available = True
except Exception as e:
    print(f"  \u26a0\ufe0f  databricks-sdk too old for tag policies ({e}); ABAC policies in step 3 will fail")
    _tag_policy_api_available = False

if _tag_policy_api_available:
    for key, values in TAG_POLICY_VALUES.items():
        try:
            _w_tags.tag_policies.create_tag_policy(
                TagPolicy(
                    tag_key=key,
                    description=f"Casper's ABAC governed tag for {key}.",
                    values=[Value(name=v) for v in values],
                )
            )
            print(f"  \u2705 governed tag policy: {key} (values: {values})")
        except Exception as e:
            msg = str(e).splitlines()[0]
            if "ALREADY_EXISTS" in msg or "already exists" in msg.lower() or "409" in msg:
                print(f"  \u267b\ufe0f  governed tag policy already exists: {key}")
            else:
                print(f"  \u26a0\ufe0f  could not create governed tag policy {key} ({type(e).__name__}): {msg}")
                print(f"     ABAC policies in step 3 referencing this key will fail until it's registered.")

# (b) Apply tags.  Wrap each in try/except so a missing table (target not
# yet materialised by Lakeflow, or skipped in a smaller target) prints and
# the loop continues.  Print the SQL on failure so the actual cause is visible.
print()
print("\u2014 Applying tags to UC securables \u2014")
applied = 0
for full_name, kind, col, tag_key, tag_value in TAG_TARGETS:
    tag_clause = f"'{tag_key}' = '{tag_value}'"
    if kind == "table":
        sql = f"ALTER TABLE {full_name} SET TAGS ({tag_clause})"
    else:
        sql = f"ALTER TABLE {full_name} ALTER COLUMN `{col}` SET TAGS ({tag_clause})"
    try:
        spark.sql(sql)
        applied += 1
        print(f"  \u2705 tagged: {full_name}{f'.{col}' if col else ''} \u2192 {tag_clause}")
    except Exception as e:
        print(f"  \u274c FAILED {full_name}{f'.{col}' if col else ''}: {str(e).splitlines()[0]}")
        print(f"     SQL: {sql}")

print(f"\n   {applied}/{len(TAG_TARGETS)} tags applied successfully")

In [ ]:
# ABAC step 2/4: UDFs that implement the row-filter / column-mask logic.
#
# These are tiny SQL functions that each ABAC policy references. Keeping the
# logic in named UDFs makes the policies declarative and lets the security
# team audit/edit the rules independently of where they're applied.
# All four are `CREATE OR REPLACE` so re-running the cell updates the body
# in place without breaking the policies that reference them.

UDFS = [
    # 1. PII column mask.  Returns the original address only to members of
    # `caspers_pii_readers`; everyone else sees `***` (workspace admins
    # always bypass row filters / column masks regardless of this UDF).
    f"""
    CREATE OR REPLACE FUNCTION {GOV_SCHEMA}.mask_pii(value STRING)
    RETURNS STRING
    COMMENT 'Mask PII unless caller is in caspers_pii_readers group.'
    RETURN CASE
      WHEN is_account_group_member('caspers_pii_readers') THEN value
      ELSE '***'
    END
    """,

    # 2. Region row filter.  Returns TRUE (row visible) if the caller is in
    # `caspers_geo_admins` OR in the region group matching the row's
    # location_code prefix. US codes: sfo/nyc/chi/lax; EMEA: lon/muc/ams/via.
    f"""
    CREATE OR REPLACE FUNCTION {GOV_SCHEMA}.filter_by_region(location_code STRING)
    RETURNS BOOLEAN
    COMMENT 'Region-based row filter: US codes for caspers_us_users, EMEA codes for caspers_emea_users, all for caspers_geo_admins.'
    RETURN
      is_account_group_member('caspers_geo_admins')
      OR (location_code IN ('sfo','nyc','chi','lax') AND is_account_group_member('caspers_us_users'))
      OR (location_code IN ('lon','muc','ams','via') AND is_account_group_member('caspers_emea_users'))
    """,

    # 3. High-value order row filter.  Orders <= $500 are visible to anyone;
    # higher-revenue orders only to caspers_finance / caspers_managers.
    f"""
    CREATE OR REPLACE FUNCTION {GOV_SCHEMA}.filter_high_value(order_revenue DOUBLE)
    RETURNS BOOLEAN
    COMMENT 'Hide orders > $500 unless caller is in caspers_finance or caspers_managers.'
    RETURN
      COALESCE(order_revenue, 0) <= 500
      OR is_account_group_member('caspers_finance')
      OR is_account_group_member('caspers_managers')
    """,

    # 4. Regulated-document row filter.  Tables tagged `caspers_sensitivity
    # = restricted` show no rows unless caller is in caspers_compliance.  This
    # is the "table-level deny" pattern — no per-row data needed, so the
    # function takes no arguments (the policy uses WHEN-clause matching).
    f"""
    CREATE OR REPLACE FUNCTION {GOV_SCHEMA}.filter_regulated_docs()
    RETURNS BOOLEAN
    COMMENT 'Hide all rows of tables tagged sensitivity=restricted unless caller is in caspers_compliance.'
    RETURN is_account_group_member('caspers_compliance')
    """,
]

for udf_sql in UDFS:
    try:
        spark.sql(udf_sql)
        # Pull function name from the SQL for the print
        name = udf_sql.split("FUNCTION", 1)[1].split("(", 1)[0].strip()
        print(f"\u2705 created/updated UDF: {name}")
    except Exception as e:
        print(f"\u26a0\ufe0f  UDF create failed: {str(e).splitlines()[0]}")

# Grant EXECUTE on each UDF to `account users` so the ABAC policies can
# actually invoke them in non-admin sessions.
for fname in ["mask_pii", "filter_by_region", "filter_high_value", "filter_regulated_docs"]:
    try:
        spark.sql(f"GRANT EXECUTE ON FUNCTION {GOV_SCHEMA}.{fname} TO `account users`")
    except Exception as e:
        print(f"\u26a0\ufe0f  GRANT EXECUTE on {fname} failed: {str(e).splitlines()[0]}")

In [ ]:
# ABAC step 3/4: declare the 4 policies.
#
# Two important shape decisions vs the earlier version:
#
#   1. `has_tag(...)` / `has_tag_value(...)` use the underscored tag keys
#      (caspers_pii etc.) that step 1 actually applies.  Dots in keys are
#      rejected by UC tag DDL — see step 1 for the verification.
#
#   2. NO `EXCEPT <group>` clauses.  `CREATE POLICY` validates EXCEPT
#      principals at creation time (PRINCIPAL_DOES_NOT_EXIST), so the
#      earlier code hard-failed against any workspace whose account
#      didn't already have the seven demo groups.  All bypass logic now
#      lives entirely inside the UDFs (step 2), which use
#      `is_account_group_member(...)` — that function returns false for
#      missing groups instead of erroring, so the policies create
#      cleanly with or without the groups.  Demo groups become truly
#      optional: present = users in those groups see unmasked data;
#      absent = mask/filter applies to everyone (workspace admins still
#      bypass everything, per UC semantics).
#
# Requires Databricks Runtime 16.4+ (CREATE POLICY syntax) on the SQL
# warehouse used by readers.

POLICIES = [
    # 1. PII column mask over any column tagged `caspers_pii=*` in the
    # simulator schema — picks up simulator.locations.address (and any
    # future PII columns added there).  Scope was `lakeflow` until we hit
    # ABAC_POLICIES_NOT_SUPPORTED on the Lakeflow-pipeline tables; see the
    # markdown intro for the full reasoning.
    (f"{CATALOG}.simulator",
     "SCHEMA",
     "caspers_mask_pii",
     f"""
     CREATE OR REPLACE POLICY caspers_mask_pii
     ON SCHEMA {CATALOG}.simulator
     COMMENT 'Mask PII columns; bypass logic lives in mask_pii UDF (caspers_pii_readers group).'
     COLUMN MASK {GOV_SCHEMA}.mask_pii
     TO `account users`
     FOR TABLES
     MATCH COLUMNS has_tag('caspers_pii') AS pii_col
     ON COLUMN pii_col
     """),

    # 2. Region row filter on any table whose column carries
    # `caspers_geo_region`.  Scoped to {GOV_SCHEMA} (where the
    # locations_snapshot lives) instead of {CATALOG}.simulator, so the
    # live simulator.locations table stays unfiltered for the ops-dashboard
    # app SP — see LOCATIONS_SNAPSHOT comment above.
    (GOV_SCHEMA,
     "SCHEMA",
     "caspers_region_filter",
     f"""
     CREATE OR REPLACE POLICY caspers_region_filter
     ON SCHEMA {GOV_SCHEMA}
     COMMENT 'Region row filter; bypass logic lives in filter_by_region UDF (caspers_geo_admins / caspers_us_users / caspers_emea_users).'
     ROW FILTER {GOV_SCHEMA}.filter_by_region
     TO `account users`
     FOR TABLES
     MATCH COLUMNS has_tag('caspers_geo_region') AS region_col
     USING COLUMNS (region_col)
     """),

    # 3. High-value-order gate on tables with a column tagged
    # `caspers_value_class = high_value`.  Targets the non-pipeline
    # snapshot table _security.high_value_orders_snapshot (CTAS from
    # lakeflow.gold_order_header) — see the markdown intro for why we
    # snapshot instead of binding to the pipeline table directly.
    (GOV_SCHEMA,
     "SCHEMA",
     "caspers_high_value_gate",
     f"""
     CREATE OR REPLACE POLICY caspers_high_value_gate
     ON SCHEMA {GOV_SCHEMA}
     COMMENT 'Hide high-value orders; bypass logic lives in filter_high_value UDF (caspers_finance / caspers_managers).'
     ROW FILTER {GOV_SCHEMA}.filter_high_value
     TO `account users`
     FOR TABLES
     MATCH COLUMNS has_tag_value('caspers_value_class', 'high_value') AS revenue_col
     USING COLUMNS (revenue_col)
     """),

    # 4. Regulated-doc deny on any table tagged
    # `caspers_sensitivity = restricted` in food_safety.
    (f"{CATALOG}.food_safety",
     "SCHEMA",
     "caspers_regulated_docs",
     f"""
     CREATE OR REPLACE POLICY caspers_regulated_docs
     ON SCHEMA {CATALOG}.food_safety
     COMMENT 'Hide restricted-tagged tables; bypass logic lives in filter_regulated_docs UDF (caspers_compliance).'
     ROW FILTER {GOV_SCHEMA}.filter_regulated_docs
     TO `account users`
     FOR TABLES
     WHEN has_tag_value('caspers_sensitivity', 'restricted')
     """),
]

created = 0
for securable, kind, name, sql in POLICIES:
    try:
        spark.sql(sql)
        created += 1
        print(f"\u2705 created policy {name} on {kind} {securable}")
    except Exception as e:
        msg = str(e).splitlines()[0]
        print(f"\u274c FAILED policy {name}: {msg}")

print(f"\n   {created}/{len(POLICIES)} policies created successfully")
print(f"   Verify via:  SHOW POLICIES IN SCHEMA {CATALOG}.simulator;")
print(f"                SHOW POLICIES IN SCHEMA {GOV_SCHEMA};")
print(f"                SHOW POLICIES IN SCHEMA {CATALOG}.food_safety;")

In [ ]:
# (Reserved cell. Previously held a fourth ABAC-step block; now removed.)
pass

#### Discover Domain tagging (DAIS 2026 beat)

Creates three governed tag policies and applies them to every UC securable
the demo materialises plus the five `all`-target AI/BI dashboards, so the
three Discover Domains (created manually in the UI per
`demos/dais2026-runbooks/SETUP.ipynb` §4) auto-surface their assets via
**Catalog → Discover**:

| Domain | Tag |
|---|---|
| Operations | `caspers_domain_operations = true` |
| Revenue & Customers | `caspers_domain_revenue = true` |
| Compliance & Safety | `caspers_domain_compliance = true` |

Boolean tag stacking lets the same asset live in multiple domains without
duplication — e.g. `lakeflow.silver_order_items` carries both
`caspers_domain_operations` and `caspers_domain_revenue` because the table
powers both operational dashboards and revenue analytics.

Keys use underscores (not dots) because the workspace-entity tag-assignment
API rejects `, . : / - =` in tag keys; UC SQL DDL accepts dots fine, but
one shared key shape lets a single Domain bind both UC tables and Genie
spaces / apps / dashboards.

Shared helpers live in `utils/domain_tags.py`.  Genie spaces and Apps are
tagged by their own stages (`Genie_Spaces`, `Databricks_App_Refund_Manager`,
`Operational_App`) so each stage owns its own assets' tags.

Same best-effort pattern as the ABAC tag block: missing securables (target
that doesn't materialise the table) or missing tag-policy admin (governed
tag creation falls back to plain SQL tags) print a warning and the cell
continues so the stage stays safe across all targets.

In [ ]:
# Discover Domain tagging.  See utils/domain_tags.py for the shared
# helpers (ensure_domain_tag_policies, tag_uc_securable,
# tag_workspace_entity).  This cell owns:
#
#   1. Governed tag policy creation (the only place we call it in the
#      job — other stages call ensure_domain_tag_policies() defensively
#      but the network call is a no-op if the policies already exist).
#   2. UC securable tagging (catalog, schemas, tables) via SQL DDL.
#   3. AI/BI dashboard tagging (all-target only) via the workspace-entity
#      tag-assignment API, looked up by name so we don't hard-code IDs.
#
# Genie spaces and Databricks Apps are tagged by their own stages so each
# stage owns its assets' tags — see stages/genie_spaces.ipynb,
# stages/apps.ipynb, stages/operational_app.ipynb.

import sys, os
sys.path.insert(0, os.path.abspath(".."))  # stages/ -> repo root
from utils.domain_tags import ensure_domain_tag_policies, tag_uc_securable, tag_workspace_entity

from databricks.sdk import WorkspaceClient
w = WorkspaceClient()

print("— Creating governed tag policies —")
ensure_domain_tag_policies(w, spark=spark)

# (full_object_name, kind, domains)   kind ∈ {"catalog", "schema", "table"}
UC_TARGETS = [
    # Catalog — surfaces in every domain so the catalog itself shows up
    # as the parent in each Domain view.
    (f"{CATALOG}", "catalog", ["operations", "revenue", "compliance"]),

    # Schemas — top-level entry points each domain user expects to see.
    (f"{CATALOG}.lakeflow",    "schema", ["operations", "revenue"]),
    (f"{CATALOG}.simulator",   "schema", ["operations"]),
    (f"{CATALOG}.food_safety", "schema", ["operations", "compliance"]),
    (f"{CATALOG}._security",   "schema", ["compliance"]),

    # Lakeflow medallion tables — granular per-table tagging so each
    # domain only surfaces the tables that matter to it.
    (f"{CATALOG}.lakeflow.all_events",                       "table", ["operations", "revenue"]),
    (f"{CATALOG}.lakeflow.silver_order_items",               "table", ["operations", "revenue"]),
    (f"{CATALOG}.lakeflow.gold_order_header",                "table", ["revenue"]),
    (f"{CATALOG}.lakeflow.gold_item_sales_day",              "table", ["revenue"]),
    (f"{CATALOG}.lakeflow.gold_brand_sales_day",             "table", ["revenue"]),
    (f"{CATALOG}.lakeflow.gold_location_sales_hourly",       "table", ["operations", "revenue"]),
    (f"{CATALOG}.lakeflow.gold_location_order_status_daily", "table", ["operations"]),

    # Simulator lookup tables.
    (f"{CATALOG}.simulator.locations",       "table", ["operations"]),
    (f"{CATALOG}.simulator.brands",          "table", ["revenue"]),
    (f"{CATALOG}.simulator.items",           "table", ["revenue"]),
    (f"{CATALOG}.simulator.menus",           "table", ["revenue"]),
    (f"{CATALOG}.simulator.brand_locations", "table", ["operations", "revenue"]),

    # AI-SQL inspection tables — compliance owns the regulated/extracted
    # version; operations also wants visibility on parsed/classified/summary.
    (f"{CATALOG}.food_safety.ai_parsed_inspections",     "table", ["operations", "compliance"]),
    (f"{CATALOG}.food_safety.ai_classified_inspections", "table", ["operations", "compliance"]),
    (f"{CATALOG}.food_safety.ai_extracted_inspections",  "table", ["compliance"]),
    (f"{CATALOG}.food_safety.ai_summarized_inspections", "table", ["operations", "compliance"]),

    # Non-pipeline snapshot table used by the high-value-order ABAC policy
    # (cell 13).  Tagged compliance (it's a governance artifact) and
    # revenue (it carries per-order revenue numbers).
    (f"{CATALOG}._security.high_value_orders_snapshot",  "table", ["compliance", "revenue"]),
]

print()
print("— Tagging UC securables —")
for full_name, kind, domains in UC_TARGETS:
    tag_uc_securable(spark, kind, full_name, domains)

# AI/BI dashboards (all-target only).  DABs deploys them, naming them
# "Casper's Kitchen - <subject> (<catalog>)" — we look up by name so we
# don't have to wire the dashboard IDs through to this cell.  Each
# dashboard is tagged with the domain(s) it primarily serves.
DASHBOARD_DOMAINS = [
    ("Casper's Kitchen - Operations",                    ["operations"]),
    ("Casper's Kitchen - Operations - Dark",             ["operations"]),
    ("Casper's Kitchen - Delivery Performance & SLA",    ["operations"]),
    ("Casper's Kitchen - AI Agent Performance",          ["operations"]),
    ("Casper's Kitchen - Menu Intelligence & Nutrition", ["revenue", "compliance"]),
]

print()
print("— Tagging AI/BI dashboards —")
# w.lakeview.list() returns Dashboard objects with `dashboard_id` and
# `display_name` populated by default (DASHBOARD_VIEW_BASIC is the API default).
try:
    all_dashboards = list(w.lakeview.list())
except Exception as e:
    print(f"  \u26a0\ufe0f  could not list dashboards ({type(e).__name__}: {e}); skipping dashboard tagging")
    all_dashboards = []

for prefix, domains in DASHBOARD_DOMAINS:
    expected_name = f"{prefix} ({CATALOG})"
    match = next((d for d in all_dashboards if getattr(d, "display_name", "") == expected_name), None)
    if match is None:
        print(f"  \u26a0\ufe0f  dashboard not found: {expected_name!r} (skipped — likely a non-all target)")
        continue
    dash_id = getattr(match, "dashboard_id", None) or getattr(match, "id", None)
    if not dash_id:
        print(f"  \u26a0\ufe0f  dashboard {expected_name!r} has no id; skipped")
        continue
    tag_workspace_entity(w, "dashboards", dash_id, domains, label=f"dashboard {expected_name!r}")

print()
print("— Environment_Helpers summary —")
print(f"  ABAC tags:        check the per-target output above (cell 11)")
print(f"  ABAC UDFs:        created in {GOV_SCHEMA} (cell 12)")
print(f"  ABAC policies:    check the per-policy output above (cell 13)")
print(f"  Discover Domain tags: check the per-target output above (cells 15-16)")
print()
print("Verify in the UI:")
print(f"  - Tags:     Catalog Explorer → {CATALOG} → (any tagged table) → Tags tab")
print(f"  - Policies: SHOW POLICIES IN SCHEMA {CATALOG}.lakeflow;")
print(f"  - Domains:  Catalog Explorer → Discover → Domains")